## Variable Definition
**Input**
1. order_t: Time interval of all orders

3. order_t_bid:[bidnumber,weightedAverageBidPriceArithmetic,GeometricMeanBidPrice,STDbid]

4. order_t_ask:[asknumber,weightedAverageAskPriceArithmetic,GeometricMeanAskPrice,STDask,spread]

5. order_t_bid_best:[BestBidPrice,BestBidQuantity]

6. order_t_ask_best:[BestAskPrice,BestAskQuantity]

7. order_t_bid_mid: [MidBidPrice,MidBidQuantity]

8. order_t_ask_mid: [MidAskPrice,MidAskQuantity]

9. price_t: [price_t_interval,trade_number]

**Target**

log_return: Volatility


## Data Prep
row1 order

row2 order

row3 order

row4 price

row5 price

row6 price

t1_order : row1-row3

t1_price : row4-row6

order block  ->  price block


In [92]:
import torch
import numpy as np
import pandas as pd
from scipy.stats import gmean
df0=df = pd.read_csv('BTCUSDT.csv')
print(df0.columns)

Index(['Datetime', 'FirstID', 'FinalID', 'Symbol', 'BidData', 'AskData',
       'Time', 'BidNumber', 'AskNumber', 'BestBidPrice', 'BestAskPrice',
       'BestBidQuantity', 'BestAskQuantity', 'Spread', 'MidBidPrice',
       'MidAskPrice', 'MidAskQuantity', 'MidBidQuantity',
       'WeightedMidBidPrice', 'WeightedMidAskPrice', 'TotalBidQuantity',
       'TotalAskQuantity', 'WeightedAverageBidPriceArithmetic',
       'WeightedAverageAskPriceArithmetic', 'GeometricMeanBidPrice',
       'GeometricMeanAskPrice', 'STDBid', 'STDAsk', 'volatility', 'id',
       'price', 'timestamp', 'log_return'],
      dtype='object')


In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import gmean
import json

symbols = ['ADAUSDT', 'BTCUSDT', 'DOGEUSDT', 'ETHUSDT', 'LTCUSDT', 'SOLUSDT', 'WBTCUSDT']
dic = {}
for symbol in symbols:
    df = pd.read_csv(f'{symbol}.csv')
    df['Datetime'] = pd.to_datetime(df['Datetime'])
    df = df.sort_values('Datetime').reset_index(drop=True)
    df['is_order'] = df['price'].isna()
    df['block_id'] = (df['is_order'] != df['is_order'].shift()).cumsum()
    first_id = df['block_id'].iloc[0]
    last_id = df['block_id'].iloc[-1]
    if not df[df['block_id'] == first_id]['is_order'].iloc[0]:
        df = df[df['block_id'] != first_id]
    if df[df['block_id'] == last_id]['is_order'].iloc[0]:
        df = df[df['block_id'] != last_id]
    grouped2 = df.groupby('block_id')
    dic2 = {}

    for block_id, group_df in grouped2:
        is_order = group_df['is_order'].iloc[0]
        start_time = group_df['Datetime'].iloc[0]
        end_time=group_df['Datetime'].iloc[-1]
        duration = (group_df['Datetime'].iloc[-1] - group_df['Datetime'].iloc[0]).total_seconds()
        
        if is_order:
            res = {
                'type': 'order',
                'start_time': start_time, 
                'duration_sec': duration,
                 'end_time':end_time,
                'bid_metrics': None, 'ask_metrics': None,
                'best_bid': [None, None], 'best_ask': [None, None],
                'mid_bid': [None, None], 'mid_ask': [None, None],
                'spread': None
            }
            
            if group_df['BidNumber'].sum() > 0:
                bn = group_df['BidNumber'].sum()
                ba = group_df['WeightedAverageBidPriceArithmetic'].mean()
                bg_data = group_df['GeometricMeanBidPrice'].dropna()
                bg = gmean(bg_data) if not bg_data.empty else np.nan
                res['bid_metrics'] = [bn, ba, bg]
                
                bbp = group_df['BestBidPrice'].max()
                bbq = group_df.loc[group_df['BestBidPrice'] == bbp, 'BestBidQuantity'].mean()
                res['best_bid'] = [bbp, bbq]
                
                bmp = group_df['MidBidPrice'].median()
                bmq = group_df.loc[group_df['MidBidPrice'] == bmp, 'MidBidQuantity'].mean()
                res['mid_bid'] = [bmp, bmq]

            if group_df['AskNumber'].sum() > 0:
                an = group_df['AskNumber'].sum()
                aa = group_df['WeightedAverageAskPriceArithmetic'].mean()
                ag_data = group_df['GeometricMeanAskPrice'].dropna()
                ag = gmean(ag_data) if not ag_data.empty else np.nan
                res['ask_metrics'] = [an, aa, ag]
                
                bap = group_df['BestAskPrice'].min()
                baq = group_df.loc[group_df['BestAskPrice'] == bap, 'BestAskQuantity'].mean()
                res['best_ask'] = [bap, baq]
                
                amp = group_df['MidAskPrice'].median()
                amq = group_df.loc[group_df['MidAskPrice'] == amp, 'MidAskQuantity'].mean()
                res['mid_ask'] = [amp, amq]
            if res['best_bid'][0] and res['best_ask'][0]:
                res['spread'] = res['best_ask'][0] - res['best_bid'][0]
            
            dic2[block_id] = res

        else:
            p_series = group_df['price']
            p_first, p_last = p_series.iloc[0], p_series.iloc[-1]
            
            dic2[block_id] = {
                'type': 'trade',
                'start_time':start_time,
                'end_time': end_time,
                'duration_sec': duration,
                'trade_stats': [len(group_df), p_series.max(), p_series.mean(), p_series.min(), p_series.median()],
                'log_return': np.log(p_last) - np.log(p_first) if p_first > 0 else 0,
            }
    dic[symbol] = pd.DataFrame.from_dict(dic2, orient='index')
final_output = {s: d.to_dict(orient='index') for s, d in dic.items()}
with open('lomndata.json', 'w') as f:
    json.dump(final_output, f, default=str)
print("JSON saved!")

JSON saved!
